# LoRA GPT-2 Medium E2E Evaluation on Google Colab

Use this notebook after training has finished and the run folder has been copied to Google Drive.

It will:

- clone this project,
- install dependencies,
- download and preprocess the E2E test data,
- load the trained LoRA adapter,
- generate the full E2E test predictions on GPU,
- compute quick BLEU and ROUGE-L metrics,
- create report figures,
- copy evaluation outputs back to Drive.

Recommended runtime: `Runtime > Change runtime type > GPU`. A faster GPU such as A100 or L4 is strongly preferred for full beam-search generation.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch

print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 2. Clone Or Update The Repo

If the repository is private, paste a GitHub token when prompted. If it is public, press Enter.

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = "justinlxiang"
REPO_NAME = "CS4782-final-project"
BRANCH = "main"
PROJECT_DIR = Path("/content") / REPO_NAME
WORK_DIR = PROJECT_DIR / "lora-gpt2-medium-e2e"

token = getpass("GitHub token, or press Enter for public clone: ")
repo_url = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
if token:
    repo_url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"

if PROJECT_DIR.exists():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print("working directory:", Path.cwd())

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Drive And Locate The Trained Adapter

The training notebook used `/content/drive/MyDrive/e2e_lora_r4_alpha32` as the default backup path. If your run folder is elsewhere, update `DRIVE_RUN_HINT` before running the cell.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

DRIVE_RUN_HINT = Path("/content/drive/MyDrive/e2e_lora_r4_alpha32")
LOCAL_RUN_DIR = Path("outputs/runs/e2e_lora_r4_alpha32")

candidates = []
for root in [DRIVE_RUN_HINT, Path("/content/drive/MyDrive")]:
    if root.exists():
        candidates.extend(root.rglob("adapter_final.pt"))

if not candidates:
    raise FileNotFoundError(
        "Could not find adapter_final.pt in Drive. Update DRIVE_RUN_HINT to your backed-up run folder."
    )

adapter_path = sorted(candidates, key=lambda path: len(path.parts))[0]
drive_run_dir = adapter_path.parents[1]
LOCAL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(drive_run_dir, LOCAL_RUN_DIR, dirs_exist_ok=True)

LOCAL_ADAPTER = LOCAL_RUN_DIR / "checkpoints" / "adapter_final.pt"
print("Drive run dir:", drive_run_dir)
print("Local adapter:", LOCAL_ADAPTER)
print("Adapter exists:", LOCAL_ADAPTER.exists())

## 5. Download And Preprocess E2E Data

In [ ]:
!mkdir -p data/raw/e2e
!curl -L -o data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!wc -l data/raw/e2e/*.txt data/processed/e2e_gpt2/*.jsonl

## 6. Generate Full Test Predictions

This is the slow step. `BATCH_SIZE=4` is conservative for beam size 10. If Colab gives you a large GPU, try `8` or `16`. If you hit out-of-memory, lower it.

In [ ]:
BATCH_SIZE = 4

!TOKENIZERS_PARALLELISM=false python scripts/generate.py \
  --config configs/e2e_gpt2_medium_lora.yaml \
  --split test \
  --adapter "$LOCAL_ADAPTER" \
  --batch-size "$BATCH_SIZE"

!ls -lh outputs/runs/e2e_lora_r4_alpha32/generations_test.txt
!python - <<'PY'
from pathlib import Path
path = Path('outputs/runs/e2e_lora_r4_alpha32/generations_test.txt')
print('prediction lines:', sum(1 for _ in path.open()))
PY

## 7. Run Metrics

In [ ]:
!python scripts/evaluate.py --config configs/e2e_gpt2_medium_lora.yaml
!cat outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json

## 9. Create Figures

In [ ]:
!python scripts/make_figures.py --config configs/e2e_gpt2_medium_lora.yaml
!ls -lh figures
!cat figures/summary.json

## 10. Back Up Evaluation Outputs To Drive

In [ ]:
import shutil
from pathlib import Path

drive_eval_dir = drive_run_dir / "evaluation_outputs"
drive_eval_dir.mkdir(parents=True, exist_ok=True)

for path in [
    Path("outputs/runs/e2e_lora_r4_alpha32/generations_test.txt"),
    Path("outputs/runs/e2e_lora_r4_alpha32/generations_test.metrics.json"),
    Path("data/processed/e2e_gpt2/references_test.txt"),
]:
    if path.exists():
        shutil.copy2(path, drive_eval_dir / path.name)

if Path("figures").exists():
    shutil.copytree("figures", drive_eval_dir / "figures", dirs_exist_ok=True)

print("Backed up evaluation outputs to:", drive_eval_dir)
!find "$drive_eval_dir" -maxdepth 2 -type f -print